# 3 · Our architecture vs. a naive offline LLM: same Qwen2.5-7B, same specs
Settings: **Accelerator: GPU T4 x2**, **Internet: On**, **Add-ons → Secrets → HF_TOKEN**.
Inputs: your code dataset **and** the output of notebook 1 (questions + PDFs).
Runs the same ~600 questions (60 judgments) two ways with the SAME model and SAME 8k window: Qwen2.5-7B given the judgment directly, and the full app. Then writes `compare/REPORT.md` and `compare/compare.png`. About 2–3 hours: *Save Version → Save & Run All*.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
LIMIT = 60          # judgments (about 10 questions each); 0 = all 300

In [ ]:
import glob, os, shutil
src = os.path.dirname(glob.glob("/kaggle/input/**/compare_naive.py", recursive=True)[0])
shutil.copytree(src, "/kaggle/working/ipc", dirs_exist_ok=True, ignore=shutil.ignore_patterns(".venv", ".cache", "__pycache__", "models", "compare"))
%cd /kaggle/working/ipc
print("code from", src)

In [ ]:
!pip install -q "transformers>=4.56,<5" peft accelerate "bitsandbytes>=0.45" huggingface_hub pymupdf matplotlib

In [ ]:
# models: your trained model + Llama-3.2-3B base + InLegalBERT, and the reranker (only the 'ours' run needs them)
!python download_model.py --repo gogulakrishnan256/hyena-llama-legal-v7
!python download_reranker.py

In [ ]:
# Ollama on GPU 1 (both systems use the same qwen2.5:7b), the app's own models on GPU 0
!apt-get install -y -qq zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '1', 'OLLAMA_FLASH_ATTENTION': '1', 'OLLAMA_KV_CACHE_TYPE': 'q8_0', 'OLLAMA_KEEP_ALIVE': '-1'}
subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/kaggle/working/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull qwen2.5:7b

In [ ]:
QA = os.path.dirname(glob.glob('/kaggle/input/**/questions.jsonl', recursive=True)[0])
os.environ.update({'CUDA_VISIBLE_DEVICES': '0', 'IPC_KEEP_RESIDENT': '1'})
OUT = '/kaggle/working/compare'
print(QA)

In [ ]:
# TRIAL (2 judgments, a few minutes): check replies look like real answers, then the full run
!python compare_naive.py $QA/questions.jsonl --system naive --ctx 8192 --limit 2 --out-dir /kaggle/working/trial
!python compare_naive.py $QA/questions.jsonl --system ours --limit 2 --out-dir /kaggle/working/trial
!python compare_naive.py --report /kaggle/working/trial

In [ ]:
# 1) naive Qwen2.5-7B, 8k window: exactly the app's specs (same model file, same 8192-token window, same GPU)
!python compare_naive.py $QA/questions.jsonl --system naive --ctx 8192 --limit $LIMIT --out-dir $OUT

In [ ]:
# OPTIONAL (off): naive Qwen2.5-7B with a 32k window - same model, but more GPU memory than the app gets.
# The main comparison is the 8k run above: same model, same 8k window as the app (IPC_NUM_CTX=8192).
# !python compare_naive.py $QA/questions.jsonl --system naive --ctx 32768 --limit $LIMIT --out-dir $OUT

In [ ]:
# 3) our architecture (the app's full pipeline, real router)
!python compare_naive.py $QA/questions.jsonl --system ours --limit $LIMIT --out-dir $OUT

In [ ]:
!python compare_naive.py --report $OUT
from IPython.display import Image, display
display(Image(OUT + '/compare.png'))